# PDPL — 9 English Sources | Local Merge + RAG Chunking

نفس منهج نسخة العربي (`PDPL_9_Sources_Local_Merge_Fixed`) لكن للمصادر الإنجليزية المستخرجة من SDAIA. يقرأ ملفات JSON التسعة من مجلد `output_json`، يوحّد الـ`document_id` مع النسخة العربية، **يقسّم حسب المادة (article)**، ثم يطبّق نفس إعدادات الـ chunking (1200 حرف / تداخل 150) ليكون الناتج متطابق البنية مع العربي.

**المخرجات** في مجلد `rag_output_en`: `pdpl_9_documents_merged_en.json` و`pdpl_9_documents_rag_chunks_en.json` و`pdpl_9_sources_report_en.csv`.

**معالجات خاصة بالإنجليزي (لا تغيّر النص القانوني، فقط البنية):**
1. ملف `sdaia_implementing_regulation.json` يحتوي وثيقتين: اللائحة التنفيذية (38 مادة) + نسخة لائحة نقل البيانات (10 مواد) بنفس الـ ids → تُفصل.
2. في `sdaia_data_transfer_regulation.json` المادة 4 مدموجة داخل المادة 3 → تُفصل لتصير 9 مواد متسلسلة.
3. السياسات الثلاث صارت مقسّمة أصلًا إلى `section` (النسخة الجديدة من الاستخراج) → تُحوَّل لنفس حقول المواد (`article_number`/`article_title`)، وتُفصل المبادئ المدموجة داخل أقسام أخرى (المبدأ الثامن في data_sharing، والمبادئ 1–6 في secondary_use) ويُحفظ البند الأب في `parent_section`.

In [ ]:
from pathlib import Path
import json, re, hashlib, unicodedata
from collections import Counter
import pandas as pd

# GitHub-portable root: run from repository root OR its notebooks/ folder.
_WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = _WORKDIR.parent if _WORKDIR.name.lower() == "notebooks" else _WORKDIR

INPUT_DIR  = PROJECT_ROOT / "output_json"      # ملفات sdaia_*.json المستخرجة (الإنجليزي)
OUTPUT_DIR = PROJECT_ROOT / "rag_output_en"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# نفس الـ document_id المستخدمة في النسخة العربية -> يسهّل المطابقة بين اللغتين
SOURCES = {
    "pdpl":                      "sdaia_pdpl.json",
    "implementing_regulation":   "sdaia_implementing_regulation.json",
    "transfer_regulation":       "sdaia_data_transfer_regulation.json",
    "data_sharing_policy":       "sdaia_data_sharing_policy.json",
    "data_monetization_policy":  "sdaia_data_monetization_policy.json",
    "appointing_dpo":            "sdaia_dpo_appointment.json",
    "accreditation_certificates":"sdaia_accreditation_certificates.json",
    "accreditation_licensing":   "sdaia_accreditation_licensing.json",
    "secondary_data_use":        "sdaia_secondary_use_of_data.json",
}
# النسخة المضمّنة من لائحة النقل داخل صفحة اللائحة التنفيذية: تُحفظ بملف جانبي افتراضيًا
INCLUDE_EMBEDDED_TRANSFER = False

print("INPUT_DIR exists:", INPUT_DIR.exists(), INPUT_DIR)
print("Output directory:", OUTPUT_DIR.resolve())

In [ ]:
def load_array(path):
    if not path.exists():
        raise FileNotFoundError(f"ملف غير موجود: {path}")
    with path.open("r", encoding="utf-8-sig") as f:
        obj = json.load(f)
    if not isinstance(obj, list) or not obj:
        raise TypeError(f"Expected non-empty JSON array in {path}")
    return obj

def ws(s):
    return re.sub(r"\s+", " ", s or "").strip()

def sha(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

raw = {doc_id: load_array(INPUT_DIR / fn) for doc_id, fn in SOURCES.items()}
for k, v in raw.items():
    print(f"{k:28s} {len(v):3d} records  unit={Counter(x['unit'] for x in v)}")

## 1) فصل الوثيقة المضمّنة من ملف اللائحة التنفيذية
تتكرر الأرقام 1–10 بعد المادة 38 بنفس الـ ids؛ هذا يعني وثيقة ثانية، فنفصلها عند أول رجوع في تسلسل رقم المادة.

In [ ]:
impl = raw["implementing_regulation"]
cut = next(i for i in range(1, len(impl)) if impl[i]["article_number"] <= impl[i-1]["article_number"])
embedded_transfer = impl[cut:]
raw["implementing_regulation"] = impl[:cut]
print("implementing_regulation:", len(raw["implementing_regulation"]), "articles | embedded transfer:", len(embedded_transfer))
print([r["article_number"] for r in raw["implementing_regulation"]][-3:], [r["article_number"] for r in embedded_transfer][:3])

## 2) فصل المواد المدموجة (مثال: المادة 4 داخل المادة 3 في لائحة النقل)
نبحث داخل نص كل مادة عن سطر عنوان مستقل بصيغة `Article N: ...` برقم مختلف عن رقم المادة.

In [ ]:
HEAD_RE = re.compile(r"(?m)^[ \t]*(Article\s+(\d+)\s*:[^\n]*)$")

def split_merged_articles(records):
    out = []
    for r in records:
        text = r["text"]
        heads = [m for m in HEAD_RE.finditer(text) if int(m.group(2)) != r["article_number"]]
        if r.get("unit") != "article" or not heads:
            out.append(r); continue
        bounds = [0] + [m.start() for m in heads] + [len(text)]
        parts = [(r["article_number"], r["article_title"], text[bounds[0]:bounds[1]])]
        for i, m in enumerate(heads):
            parts.append((int(m.group(2)), m.group(1).strip(), text[m.end():bounds[i+2]]))
        for num, title, body in parts:
            item = dict(r)
            item.update(article_number=num, article_title=title, text=body.strip(),
                        quality_flags=sorted(set(r.get("quality_flags", [])) | {"split_from_merged_article"}))
            item["clean_text"] = ws(body)
            out.append(item)
        print(f"split {r['id']} -> articles {[p[0] for p in parts]}")
    return out

for k in list(raw):
    raw[k] = split_merged_articles(raw[k])
embedded_transfer = split_merged_articles(embedded_transfer)
print("transfer_regulation articles:", [r["article_number"] for r in raw["transfer_regulation"]])

## 3) السياسات الثلاث (`section`)
الاستخراج الجديد قسّم السياسات إلى أقسام (`section_number`/`section_title`). نحوّلها لنفس حقول المواد، ونفصل عناوين المبادئ المدموجة داخل نص قسم آخر، ونرقّم الأقسام من جديد بتسلسل. المبادئ تحتفظ بالبند الأب في `parent_section`.

In [ ]:
PRINC = r"(?:(?:First|Second|Third|Fourth|Fifth|Sixth|Seventh|Eighth?|Ninth|Tenth) Principle|Principle \d+)"
PRINCIPLE_RE       = re.compile(rf"(?m)^[ \t]*({PRINC}):[ \t]*([^\n]*)$")   # heading line inside a text
PRINCIPLE_TITLE_RE = re.compile(rf"^{PRINC}:")                                  # a section title
# البند الأب للمبادئ (من تركيب الصفحة الأصلية)
PRINCIPLE_PARENT = {
    "data_sharing_policy":      "Second: The Main Principles for Data Sharing",
    "data_monetization_policy": "Fourth: Basic Principles of Data Monetization",
    "secondary_data_use":       "Fourth: Principles of Secondary Use of Data",
}

def normalize_sections(doc_id, records):
    items = []
    for r in records:
        text = r["text"]
        heads = [m for m in PRINCIPLE_RE.finditer(text) if m.start() > 0]
        bounds = [0] + [m.start() for m in heads] + [len(text)]
        parts = [(r["section_title"], text[bounds[0]:bounds[1]])]
        parts += [(f"{m.group(1)}: {m.group(2).strip()}", text[m.end():bounds[i+2]]) for i, m in enumerate(heads)]
        for title, body in parts:
            item = dict(r)
            item.update(section_title=title, text=body.strip(), clean_text=ws(body), text_sha256=sha(body.strip()))
            if heads:
                item["quality_flags"] = sorted(set(r.get("quality_flags", [])) | {"split_from_merged_section"})
            items.append(item)
        if heads:
            print(f"split {r['id']} -> {[t for t, _ in parts]}")
    for i, item in enumerate(items, 1):                      # sequential renumbering
        item["section_number"] = i
        item["article_number"] = i
        item["article_title"] = item["section_title"]
        if PRINCIPLE_TITLE_RE.match(item["section_title"]):
            item["parent_section"] = PRINCIPLE_PARENT[doc_id]
    return items

for k in ("data_sharing_policy", "data_monetization_policy", "secondary_data_use"):
    assert all(r["unit"] == "section" for r in raw[k]), f"{k}: expected pre-split sections"
    raw[k] = normalize_sections(k, raw[k])
    print(k, len(raw[k]), "sections")
    for r in raw[k]:
        print("   ", r["article_number"], "|", r["article_title"][:60], "|", len(r["clean_text"]))

## 4) Merge (نفس منطق العربي: التحقق من الـ ids، منع التكرار المتعارض، الاحتفاظ بـ `needs_review`)

In [ ]:
def prepare(doc_id, r):
    item = dict(r)
    item["source_document_id"] = item.get("document_id")
    item["document_id"] = doc_id
    item["language"] = "en"
    item.setdefault("parent_section", None)
    item["id"] = f"{doc_id}_{int(item['article_number']):03d}"
    item["clean_text"] = item.get("clean_text") or item.get("text") or ""
    item.setdefault("extraction_status", "needs_review")
    return item

core_records = [prepare(d, r) for d, recs in raw.items() for r in recs]
embedded_records = []
for r in embedded_transfer:
    e = prepare("transfer_regulation_embedded_v1", r)
    e["quality_flags"] = sorted(set(e.get("quality_flags", [])) | {"embedded_in_implementing_regulation_page"})
    embedded_records.append(e)

records_in = core_records + (embedded_records if INCLUDE_EMBEDDED_TRANSFER else [])

# Merge defensively: duplicate IDs with different content are an error.
merged_records, by_id = [], {}
for original in records_in:
    record = dict(original)
    rid = record.get("id")
    if not isinstance(rid, str) or not rid.strip():
        raise ValueError(f"Record without valid id: {record.get('document_id')}")
    if not record["clean_text"].strip():
        raise ValueError(f"Empty legal text: {rid}")
    if rid in by_id:
        if by_id[rid] != record:
            raise ValueError(f"Conflicting duplicate id: {rid}")
        continue
    by_id[rid] = record
    merged_records.append(record)

counts = Counter(r["document_id"] for r in merged_records)
EXPECTED_DOCS = set(SOURCES) | ({"transfer_regulation_embedded_v1"} if INCLUDE_EMBEDDED_TRANSFER else set())
if set(counts) != EXPECTED_DOCS:
    raise AssertionError(f"Expected {len(EXPECTED_DOCS)} docs. Missing: {EXPECTED_DOCS-set(counts)}")

# article numbers must be contiguous within each article-based document
for d in SOURCES:
    nums = [r["article_number"] for r in merged_records if r["document_id"] == d]
    if nums: assert nums == list(range(1, len(nums)+1)), f"Gap in {d}: {nums}"

report = pd.DataFrame([{"document_id": k, "records": v} for k, v in sorted(counts.items())])
display(report)
print("Documents:", len(counts), "| Records:", len(merged_records))

merged_path = OUTPUT_DIR / "pdpl_9_documents_merged_en.json"
with merged_path.open("w", encoding="utf-8") as f:
    json.dump(merged_records, f, ensure_ascii=False, indent=2)

side_path = OUTPUT_DIR / "embedded_transfer_regulation_v1_en.json"
with side_path.open("w", encoding="utf-8") as f:
    json.dump(embedded_records, f, ensure_ascii=False, indent=2)
print("Saved:", merged_path.resolve())
print("Side file (embedded transfer regulation):", side_path.resolve(), len(embedded_records), "articles")

## Chunking (حسب المادة)
كل مادة/بند = وحدة أساسية. المادة التي تقل عن 1200 حرف تبقى chunk واحد، والأطول تُقسّم بحد أقصى تقريبي 1200 حرف وتداخل 150 مع الاحتفاظ بـ`parent_id` ورقم المادة (نفس إعدادات العربي). **خطوة تقنية ولا تُعدّ تحققًا قانونيًا.**

In [ ]:
MAX_CHARS = 1200
OVERLAP = 150

def split_preserving_words(s, max_chars=MAX_CHARS, overlap=OVERLAP):
    s = re.sub(r"\s+", " ", s).strip()
    if not s: return []
    pieces, start = [], 0
    while start < len(s):
        end = min(len(s), start + max_chars)
        if end < len(s):
            space = s.rfind(" ", start + max_chars // 2, end)
            if space > start:
                end = space
        part = s[start:end].strip()
        if part: pieces.append(part)
        if end >= len(s): break
        start = max(start + 1, end - overlap)
    return pieces

def normalize_en_search(txt):
    txt = unicodedata.normalize("NFKC", txt)
    txt = txt.translate(str.maketrans({"\u2018":"'", "\u2019":"'", "\u201c":'"', "\u201d":'"',
                                       "\u2013":"-", "\u2014":"-", "\u00a0":" "}))
    return re.sub(r"\s+", " ", txt).strip().lower()

rag_chunks = []
for r in merged_records:
    for j, piece in enumerate(split_preserving_words(r["clean_text"]), 1):
        rag_chunks.append({
            "chunk_id": f"{r['id']}_chunk_{j:02d}",
            "parent_id": r["id"],
            "document_id": r["document_id"],
            "language": "en",
            "unit": r.get("unit"),
            "article_number": r.get("article_number"),
            "article_title": r.get("article_title"),
            "parent_section": r.get("parent_section"),
            "source_url": r.get("source_url"),
            "extraction_status": r.get("extraction_status", "needs_review"),
            "clean_text": piece,
            "search_text": normalize_en_search(piece),
        })

assert len({c["chunk_id"] for c in rag_chunks}) == len(rag_chunks)
chunk_path = OUTPUT_DIR / "pdpl_9_documents_rag_chunks_en.json"
with chunk_path.open("w", encoding="utf-8") as f:
    json.dump(rag_chunks, f, ensure_ascii=False, indent=2)

chunk_stats = (pd.DataFrame(rag_chunks)
    .assign(length=lambda df: df.clean_text.str.len())
    .groupby("document_id", as_index=False)
    .agg(chunks=("chunk_id", "size"), min_length=("length", "min"), max_length=("length", "max")))
display(chunk_stats)
print("Total chunks:", len(rag_chunks), "| Unique documents:", len(chunk_stats))
print("Saved:", chunk_path.resolve())

In [ ]:
# Final report and safety gates
final = report.merge(chunk_stats, on="document_id", how="left", validate="one_to_one")
final_path = OUTPUT_DIR / "pdpl_9_sources_report_en.csv"
final.to_csv(final_path, index=False, encoding="utf-8-sig")

assert len(final) == len(EXPECTED_DOCS), "Incomplete merge!"
assert sum(final["chunks"]) == len(rag_chunks)
assert all("source_url" in r for r in merged_records), "Some source URLs missing as keys"
assert all(c["clean_text"].strip() for c in rag_chunks)

print("✅ Merge completed with nine English sources")
print("Merged:", merged_path.resolve())
print("Chunks:", chunk_path.resolve())
print("Report:", final_path.resolve())
print("Review markers retained; do not represent records as source-verified.")

In [ ]:
# Sanity check: reload the chunks file
with open(OUTPUT_DIR / "pdpl_9_documents_rag_chunks_en.json", encoding="utf-8") as f:
    chunks = json.load(f)
counts_c = Counter(c["document_id"] for c in chunks)
print("Total documents:", len(counts_c))
print("Total chunks:", len(chunks))
print("Missing core docs:", [d for d in ["pdpl","implementing_regulation","transfer_regulation"] if d not in counts_c])
for doc, n in sorted(counts_c.items()):
    print(f"{doc}: {n}")